# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:

print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 282.35 GB
MemAvailable: 980.82 GB
Free GPU Memory (GB): 39.3936

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################



/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################



## 2. Loading Datasets

### 2.1 T-Rex

In [ ]:
import os

print("\n################################")
print("Setting up T-REX...")
print("################################\n")

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.

from datasets import load_dataset
ds = load_dataset("relbert/t_rex")
ds


## 3. FKTC Evaluation

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import os
import pandas as pd

class ResponseGenerator:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
        self.model.eval()

    def generate_response(self, query, strategy, true_answer, max_new_tokens, temperature):
        prompt = self.get_prompt(query, strategy)
        input_ids = self.tokenizer.encode(prompt, return_tensors='pt').to("cuda")
        generation_config = {
            "temperature": temperature,
            "do_sample": True,
            "top_p": 0.75,
            "top_k": 40,
            "num_beams": 5,
            "num_return_sequences": 3,
            "output_scores": True,
            "output_hidden_states": False,
            "output_attentions": False,
            "return_dict_in_generate": True
        }

        with torch.no_grad():
            outputs = self.model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens)

        output_text = self.tokenizer.decode(outputs[0][0], skip_special_tokens=True)
        output_text = self.clean_response(output_text, strategy)
        response_tokens = outputs[0].tolist()
        output_scores = outputs.scores

        is_correct, cumulative_prob, beams_with_probs, top_tokens_with_probs = self.check_answer(response_tokens[0], true_answer, output_scores, input_ids)
        return output_text, is_correct, cumulative_prob, beams_with_probs, top_tokens_with_probs

    def get_prompt(self, query, strategy):
        # Define the prompt formatting based on the selected strategy
        prompt_strategies = {
            "Fact Statement": f"{query} Fact:",
            "Completion": f"{query} The answer is:",
            "Definitive Statement": f"The answer to the question '{query}' is:",
            "True Statement": f"It is true that the answer to '{query}' is:",
            "Declarative Statement": f"{query} The fact is:",
            "Conclusive Statement": f"The final answer to '{query}' is:",
            "Resolved Statement": f"Resolved: '{query}' The answer is:",
            "Ending Completion": f"{query} The final answer is:",
            "Answer Completion": f"{query} The correct answer is:",
            "Plain Completion": f"{query} The answer:",
            "Direct Completion": f"{query} Answer:",
            "Simple Completion": f"{query} Result:",
            "Direct Answer": f"{query} Correct answer:",
            "Answer Statement": f"{query} The exact answer is:",
            "True Completion": f"{query} The true answer is:"
        }
        return prompt_strategies.get(strategy, query)

    def clean_response(self, output_text, strategy):
        # Clean the output text based on the selected strategy
        strategy_endings = {
            "Fact Statement": "Fact:",
            "Completion": "The answer is:",
            "Definitive Statement": "is:",
            "True Statement": "is:",
            "Declarative Statement": "The fact is:",
            "Conclusive Statement": "is:",
            "Resolved Statement": "is:",
            "Ending Completion": "is:",
            "Answer Completion": "The correct answer is:",
            "Plain Completion": "The answer:",
            "Direct Completion": "Answer:",
            "Simple Completion": "Result:",
            "Direct Answer": "Correct answer:",
            "Answer Statement": "The exact answer is:",
            "True Completion": "The true answer is:"
        }
        return output_text.split(strategy_endings.get(strategy, ""))[-1].strip()

    def check_answer(self, response_tokens, true_answer, output_scores, input_ids):
        generated_tokens = response_tokens[input_ids.size(1):]
        if len(generated_tokens) == 0:
            return False, None, None, None
        assert len(generated_tokens) == len(output_scores)
        
        # Generate the four variants of true_answer
        true_answer_lower = true_answer.lower()
        true_answer_title = true_answer.title()
        true_tokens_no_space_lower = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(true_answer_lower))
        true_tokens_no_space_title = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(true_answer_title))
        true_tokens_with_space_lower = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(" " + true_answer_lower))
        true_tokens_with_space_title = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(" " + true_answer_title))

        variants = [
            true_tokens_no_space_lower,
            true_tokens_no_space_title,
            true_tokens_with_space_lower,
            true_tokens_with_space_title
        ]

        def get_cumulative_probability(true_tokens, idx, output_scores):
            cumulative_prob = 1.0
            top_tokens_with_probs = []
            beams_with_probs = []

            for true_token_idx, true_token in enumerate(true_tokens):
                token_probs = []
                for beam_index, beam_scores in enumerate(output_scores[idx + true_token_idx]):
                    token_probs = torch.softmax(beam_scores, dim=-1)
                    token_prob = token_probs[true_token].item()
                    top_indices = (token_probs >= 0.1).nonzero(as_tuple=True)[0]
                    top_probs = token_probs[top_indices]
                    top_tokens = self.tokenizer.convert_ids_to_tokens(top_indices)
                    top_tokens = [token.replace("Ġ", " ") for token in top_tokens]

                    if true_token in top_indices:
                        top_tokens_with_probs.extend([(token, prob.item()) for token, prob in zip(top_tokens, top_probs)])
                        cumulative_prob *= token_prob
                        beams_with_probs.append({
                            'character_index': idx + true_token_idx,
                            'beam_index': beam_index,
                            'token': self.tokenizer.decode([true_token]),
                            'token_index': true_token,
                            'probability': token_prob
                        })
                        break

            return cumulative_prob, beams_with_probs, top_tokens_with_probs

        # Check each variant
        for variant in variants:
            for idx in range(len(generated_tokens) - len(variant) + 1):
                if generated_tokens[idx:idx + len(variant)] == variant:
                    cumulative_prob, beams_with_probs, top_tokens_with_probs = get_cumulative_probability(variant, idx, output_scores)
                    return True, cumulative_prob, beams_with_probs, top_tokens_with_probs

        return False, None, None, None

In [ ]:
strategies = [
    "Fact Statement", "Completion", "Definitive Statement", "True Statement",
    "Declarative Statement", "Conclusive Statement", "Resolved Statement",
    "Ending Completion", "Answer Completion", "Plain Completion", "Direct Completion",
    "Simple Completion", "Direct Answer", "Answer Statement", "True Completion"
]

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import os

class MonitorEvaluator:
    def __init__(self, model_name, data_dir, files, generation_config=None, max_new_tokens=15, verbose=True):
        print("Initializing MonitorEvaluator...")
        print(f"Loading model {model_name}...")
        print(f"Loading tokenizer {model_name}...")
        print(f"Loading data from {data_dir}...")
        print(f"Loading files {files}...")
        print(f"Max new tokens: {max_new_tokens}")
        self.generator = ResponseGenerator(model_name)
        self.data_dir = data_dir
        self.files = files
        self.max_new_tokens = max_new_tokens
        self.verbose = verbose
        self.generation_config = generation_config

    def load_json_data(self, filename):
        if self.verbose:
            print(f"Loading data from {filename}...")
        with open(os.path.join(self.data_dir, filename), 'r', encoding='utf8') as f:
            return [json.loads(line) for line in f.readlines()[:4]]  # Load only 4 lines per file for testing

    def evaluate(self):
        if self.verbose:
            print("Evaluating FKTC data...")
        all_results = []
        for file in self.files[:3]:  # Limit to 3 files for testing
            data = self.load_json_data(file)
            relations = data[0]['relations']
            for idx, entry in enumerate(data[1:3]):
                subject = entry['subject']
                true_object = entry['object']
                taxonomy = entry['taxonomy']
                results = []

                for relation in relations[:1]:
                    for strategy in [
                        "Fact Statement", "Completion", "Definitive Statement", "True Statement",
                        "Declarative Statement", "Conclusive Statement", "Resolved Statement",
                        "Ending Completion", "Answer Completion", "Plain Completion", "Direct Completion",
                        "Simple Completion", "Direct Answer", "Answer Statement", "True Completion"
                    ]:
                        # Evaluate original relation
                        prompt = relation.replace("[X]", subject)
                        output_text, is_correct, cumulative_prob, beams_with_probs, top_tokens_with_probs = self.generator.generate_response(prompt, strategy, true_object, self.max_new_tokens, self.generation_config.temperature)
                        results.append({
                            'file': file,
                            'entry': idx,
                            'relation': relation,
                            'prompt': prompt,
                            'subject': subject,
                            'true_object': true_object,
                            'strategy': strategy,
                            'output_text': output_text,
                            'is_correct': is_correct,
                            'cumulative_prob': cumulative_prob,
                            'beams_with_probs': beams_with_probs,
                            'top_tokens_with_probs': top_tokens_with_probs
                        })

                all_results.append(results)
        return all_results

    def save_results(self, results, output_file):
        with open(output_file, 'w', encoding='utf8') as f:
            json.dump(results, f, indent=4)

if __name__ == "__main__":
    model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    # model_name = "TinyLlama/TinyLlama_v1.1"
    # model_name = "bigscience/bloomz-560m"
    model_name = "bigscience/bloomz-1b1"
    model_name = "meta-llama/Meta-Llama-3-8B"
    # model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
    data_dir = "/nfs/students/daro/data/MONITOR/FKTC"
    files = [
        "P101-subclass.json",
        "P103-subclass.json",
        "P108-subclass.json",
        "P127-subclass.json",
        "P1376-subclass.json",
        "P1412-subclass.json",
        "P159-subclass.json",
        "P17-subclass.json",
        "P176-subclass.json",
        "P178-subclass.json",
        "P19-subclass.json",
        "P20-subclass.json",
        "P264-subclass.json",
        "P27-subclass.json",
        "P276-subclass..json",
        "P30-subclass.json",
        "P364-subclass.json",
        "P37-subclass.json",
        "P495-subclass.json",
        "P740-subclass.json"
    ]
    output_file = "evaluation_results.json"
    max_new_tokens = 15
    generation_config = {
        "temperature": 0.1,
        "top_p": 0.75,
        "top_k": 40,
        "num_beams": 5,
        "num_return_sequences": 1,
        "output_scores": True,
        "output_hidden_states": False,
        "output_attentions": False,
        "return_dict_in_generate": True
    }
    evaluator = MonitorEvaluator(
        model_name=model_name,
        data_dir=data_dir,
        files=files,
        generation_config=generation_config,
        max_new_tokens=max_new_tokens,
        verbose=False
    )
    results = evaluator.evaluate()
    for results_list in results:
        for result in results_list:
            if True:
                print(f"Prompt: {result['prompt']}")
                print(f"Answer: {result['answer']}")
                print(f"True Object: {result['true_object']}")
    print(f"Correct answers: {sum([sum([result['is_correct'] for result in results_list]) for results_list in results])}")
    print(f"Incorrect answers: {sum([sum([not result['is_correct'] for result in results_list]) for results_list in results])}")
    evaluator.save_results(results, output_file)

In [ ]:
import json
import os

class FKTCObjectChecker:
    def __init__(self, data_dir, files):
        self.data_dir = data_dir
        self.files = files

    def load_json_data(self, filename):
        with open(os.path.join(self.data_dir, filename), 'r', encoding='utf8') as f:
            return [json.loads(line) for line in f.readlines()]

    def check_objects_for_multiple_words(self):
        multi_word_objects = {}
        for file in self.files:
            data = self.load_json_data(file)
            for entry in data:
                obj = entry.get('object', "")
                if len(obj.split()) > 1:  # Check if the object contains more than one word
                    if file not in multi_word_objects:
                        multi_word_objects[file] = []
                    multi_word_objects[file].append(obj)
        
        return multi_word_objects

if __name__ == "__main__":
    data_dir = "/nfs/students/daro/data/MONITOR/FKTC"
    files = [
        "P101-subclass.json",
        "P103-subclass.json",
        "P108-subclass.json",
        "P127-subclass.json",
        "P1376-subclass.json",
        "P1412-subclass.json",
        "P159-subclass.json",
        "P17-subclass.json",
        "P176-subclass.json",
        "P178-subclass.json",
        "P19-subclass.json",
        "P20-subclass.json",
        "P264-subclass.json",
        "P27-subclass.json",
        "P276-subclass..json",
        "P30-subclass.json",
        "P364-subclass.json",
        "P37-subclass.json",
        "P495-subclass.json",
        "P740-subclass.json"
    ]

    checker = FKTCObjectChecker(data_dir, files)
    multi_word_objects = checker.check_objects_for_multiple_words()

    if multi_word_objects:
        print("Files with multi-word 'object' values:")
        for file, objects in multi_word_objects.items():
            print(f"\n{file}:")
            for obj in objects:
                print(f"  - {obj}")
    else:
        print("No multi-word 'object' values found in the files.")

In [ ]:
results = evaluator.evaluate()
for results_list in results:
    for result in results_list:
        if result['is_correct']:
            print(f"Prompt: {result['prompt']}")
            print(f"Answer: {result['answer']}")
            print(f"True Object: {result['true_object']}")

In [ ]:
print(sum([sum([result['is_correct'] for result in results_list]) for results_list in results]))
print(sum([sum([not result['is_correct'] for result in results_list]) for results_list in results]))

In [ ]:
for results_list in results:
  for result in results_list:
    if not result['is_correct']:
      print(f"Prompt: {result['prompt']}")
      print(f"Answer: {result['answer']}")
      print(f"True Object: {result['true_object']}")

In [ ]:
import transformers
import torch

model_id = "meta-llama/Meta-Llama-3-8B"
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
pipeline = transformers.pipeline(
  "text-generation", model=model_id, model_kwargs={"torch_dtype": torch.bfloat16}, device_map="cuda"
)
# output = pipeline("What is the capital city of Hungary?", max_new_tokens=15)
output = pipeline("Which city is Chandos Records's corporate headquarters located?", max_new_tokens=100)
answer = output[0]['generated_text']
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(output)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Load tokenizer and model with float16 precision
print("Loading model...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Create a pipeline for text generation
print("Creating pipeline...")
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="cuda")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Perform inference with the query
print("Performing inference...")
query = "Which city is Chandos Records's corporate headquarters located?"
# query = "What is the capital city of Hungary?"
output = generator(query, max_length=200, num_return_sequences=1)
print(output)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import numpy as np

print("Loading tokenizer and model...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
# model_name = "bigscience/bloomz-560m"
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# model_name = "TinyLlama/TinyLlama_v1.1"
# model_name = "bigscience/bloomz-560m"
model_name = "bigscience/bloomz-1b1"
# model_name = "meta-llama/Meta-Llama-3-8B"
# model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")

# query = "Which city is Eiffel Tower located in?"
# query = "Spanish. What is the native language of Louis Jules Trochu?"
query = "Which industry does Alan Turing work in?"
# query = "What is the location of Simcoe Composite School?"
input_ids = tokenizer.encode(query, return_tensors='pt').to("cuda")

max_length = 50
generation_config = {
    "temperature": 1,
    "top_p": 0.75,
    "top_k": 40,
    "num_beams": 5,
    "num_return_sequences": 1,
    "output_scores": True,
    "output_hidden_states": False,
    "output_attentions": False,
    "return_dict_in_generate": True
}

print("Generating output...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
with torch.no_grad():
    output_ids = model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=15)

print("Decoding output...")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
output_text = tokenizer.decode(output_ids[0][0], skip_special_tokens=True)
print(output_text)

In [ ]:
from transformers import AutoTokenizer
import transformers 
import torch
model = "TinyLlama/TinyLlama_v1.1"
# model = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model)
pipeline = transformers.pipeline(
    "text-generation",
    model=model,
    torch_dtype=torch.float16,
    device_map="auto",
)

sequences = pipeline(
    query,
    do_sample=True,
    top_k=10,
    num_return_sequences=1,
    repetition_penalty=1.5,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=15,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")


### 3.1 All strategies

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd

class ResponseGenerator:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
        self.model.eval()

    def generate_response(self, prompt, max_new_tokens, temperature):
        input_ids = self.tokenizer.encode(prompt, return_tensors='pt').to("cuda")
        generation_config = {
            "temperature": temperature,
            "do_sample": True,
            "top_p": 0.75,
            "top_k": 40,
            "num_beams": 5,
            "num_return_sequences": 1,
            "output_scores": True,
            "output_hidden_states": False,
            "output_attentions": False,
            "return_dict_in_generate": True
        }

        with torch.no_grad():
            outputs = self.model.generate(input_ids, generation_config=GenerationConfig(**generation_config), max_new_tokens=max_new_tokens)

        output_text = self.tokenizer.decode(outputs[0][0], skip_special_tokens=True)
        probabilities = self.extract_probabilities(outputs)
        return output_text, probabilities

    def extract_probabilities(self, outputs):
        probabilities = []
        for score in outputs.scores:
            probs = torch.softmax(score[0], dim=-1)
            top_prob, top_idx = torch.max(probs, dim=-1)
            probabilities.append((self.tokenizer.decode(top_idx), top_prob.item()))
        return probabilities

def apply_prompt_strategy(query, strategy):
    if strategy == "Direct Instruction":
        return f"Please answer the following question in one word.\nQuestion: {query}\nAnswer:"
    elif strategy == "Contextual Prompts":
        return f"{query} (Please answer in one word)"
    elif strategy == "Explicit Formatting":
        return f"What is the location of {query}?\nAnswer (one word):"
    elif strategy == "Question-Answer Pairs":
        return f"QSTN: What is the capital of France?\nANSR: Paris\nQSTN: What is the capital of Germany?\nANSR: Berlin\nQSTN: {query}\nANSR:"
    elif strategy == "Negative Examples":
        return f"QSTN: What is the capital of France?\nANSR: Berlin (incorrect)\nANSR: Paris (correct)\nQSTN: {query}\nANSR:"
    elif strategy == "Direct Answer Request":
        return f"Give a concise answer: {query}\nAnswer:"
    elif strategy == "Role Play":
        return f"You are a geography expert known for your concise answers.\nQuestion: {query}\nAnswer:"
    elif strategy == "Simplified Question":
        return f"Where is {query} located?\nAnswer:"
    elif strategy == "List Format":
        return f"List of schools and their locations:\n1. Harvard University - USA\n2. University of Cambridge - UK\n3. {query} -"
    elif strategy == "Multiple Choice":
        return f"Select the correct location of {query}:\nA) USA\nB) UK\nC) Canada\nAnswer:"
    elif strategy == "Fill-in-the-Blank":
        return f"{query} is located in _____.\nAnswer:"
    elif strategy == "Structured Answer Prompt":
        return f"Question: {query}\nAnswer (one word):"
    else:
        return query

# Example usage:
model_names = [
    "bigscience/bloomz-560m",
    "bigscience/bloomz-1b1",
    "openai-community/gpt2-large",
    "EleutherAI/gpt-neo-1.3B",
    "TinyLlama/TinyLlama_v1.1",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    #"meta-llama/Meta-Llama-3-8B",
    #"meta-llama/Meta-Llama-3-8B-Instruct",
]
queries = [
    "What is the location of Simcoe Composite School?",
    "What is Alan Turing's area of expertise?",
    "What is the native language of Louis Jules Trochu?",
    "What is the capital city of Italy?",
    "What is the main ingredient in sushi?"
]
true_answers = [
    "Canada",
    "logic",
    "French",
    "Rome",
    "rice"
]
max_new_tokens_list = [5, 15, 25]
temperature_list = [0.1, 1]
strategies = [
    # "Direct Instruction",
    # "Contextual Prompts",
    # "Explicit Formatting",
    "Question-Answer Pairs",
    "Negative Examples",
    # "Direct Answer Request",
    # "Role Play",
    # "Simplified Question",
    # "List Format",
    # "Multiple Choice",
    # "Fill-in-the-Blank",
    # "Structured Answer Prompt"
]

# Initialize a list to store the results
results = []

for model_name in model_names:
    generator = ResponseGenerator(model_name)
    for query, true_answer in zip(queries, true_answers):
        for max_new_tokens in max_new_tokens_list:
            for temperature in temperature_list:
                for strategy in strategies:
                    print(f"Model: {model_name}, Query: {query}, Strategy: {strategy}, Max New Tokens: {max_new_tokens}, Temperature: {temperature}")
                    prompt = apply_prompt_strategy(query, strategy)
                    output_text, probabilities = generator.generate_response(prompt, max_new_tokens, temperature)
                    if output_text.startswith(prompt):
                        output_text = output_text[len(prompt):].strip()
                    results.append({
                        "Model": model_name,
                        "Query": query,
                        "Strategy": strategy,
                        "Max New Tokens": max_new_tokens,
                        "Temperature": temperature,
                        "True Answer": true_answer,
                        "Output": output_text,
                        "Probabilities": probabilities
                    })

# Convert the results list to a pandas DataFrame
df = pd.DataFrame(results)

# Print the DataFrame
print(df)

# Export the DataFrame to an Excel file
df.to_excel("results_07_25_qa_negative.xlsx", index=False)

### 3.3 Cleaned

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd
import numpy as np

def get_prompt(query, strategy):
    if strategy == "Fact Statement":
        prompt = f"{query} Fact:"
    elif strategy == "Completion":
        prompt = f"The answer is:"
    elif strategy == "Definitive Statement":
        prompt = f"The answer to the question '{query}' is:"
    # Add other strategies if needed
    return prompt

def clean_response(output_text, strategy):
    if strategy == "Fact Statement":
        output_text = output_text.split("Fact:")[-1].strip()
    elif strategy == "Completion":
        output_text = output_text.split("The answer is:")[-1].strip()
    elif strategy == "Definitive Statement":
        output_text = output_text.split("is:")[-1].strip()
    # Add other strategies if needed
    return output_text

def calculate_entropy(probs):
    return -np.sum(probs * np.log(probs))

class ResponseGenerator:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
        
        # Set pad_token_id to eos_token_id to avoid the warning
        self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
        
        self.model.eval()

    def generate_response(self, query, strategy, true_answer, max_new_tokens, temperature):
        prompt = get_prompt(query, strategy)
        inputs = self.tokenizer(prompt, return_tensors='pt').to("cuda")
        
        generation_config = {
            "temperature": temperature,
            "do_sample": True,  # Enable multinomial sampling
            "top_p": 0.75,
            "top_k": 40,
            "output_scores": True,
            "output_hidden_states": False,
            "output_attentions": False,
            "return_dict_in_generate": True,
            "pad_token_id": self.tokenizer.eos_token_id  # Set pad_token_id to avoid warning
        }

        with torch.no_grad():
            outputs = self.model.generate(
                inputs['input_ids'],
                attention_mask=inputs['attention_mask'],  # Provide attention_mask to avoid the warning
                generation_config=GenerationConfig(**generation_config),
                max_new_tokens=max_new_tokens
            )

        output_text = self.tokenizer.decode(outputs[0][0], skip_special_tokens=True)
        output_text = clean_response(output_text, strategy)
        response_tokens = outputs.sequences[0].tolist()

        # Calculate probabilities for the generated part
        probs = []
        token_probs = []
        sequence = outputs.sequences[0]
        cum_prob = 1.0
        shift_idx = len(sequence) - len(outputs.scores)  # Position to start processing tokens
        for pos_idx, beam_scores in enumerate(outputs.scores):
            softmax_scores = torch.softmax(beam_scores, dim=-1)
            token_id = sequence[pos_idx + shift_idx]
            token_prob = softmax_scores[0][token_id].item()
            token = self.tokenizer.decode([token_id])
            token_probs.append((token, round(token_prob, 4)))
            cum_prob *= token_prob
            probs.append(token_prob)
        
        probs = torch.tensor(probs)
        adj_prob = torch.exp((len(probs) ** -1) * torch.sum(torch.log(probs)))
        
        # Calculate entropy score
        entropy = calculate_entropy(np.array(probs))

        # Check if the generated answer is correct
        is_correct = true_answer.lower() in output_text.lower()

        return output_text, cum_prob, adj_prob, entropy, token_probs, is_correct

# Example usage:
model_names = ["meta-llama/Meta-Llama-3-8B"]
queries = [
    "What is the location of Simcoe Composite School?",
    "musical. What is Alan Turing's area of expertise?",
    "Latin. What is the native language of Louis Jules Trochu?",
    "What is the top speed of a Formula 1 car?",
    "What is the main ingredient of a Caesar salad?",
    "What is the capital city of France?",
    "Who discovered penicillin?",
    "What language is primarily spoken in Brazil?",
    "What is the primary spice used in curry?",
    "Who wrote the play 'Hamlet'?",
    "What is the primary color of bananas?",
    "Who is the author of '1984'?"
]
true_answers = [
    "Canada",
    "logic",
    "French",
    "360 km/h",
    "lettuce",
    "Paris",
    "Alexander Fleming",
    "Portuguese",
    "Turmeric",
    "William Shakespeare",
    "yellow",
    "George Orwell"
]
max_new_tokens_list = [15, 25]
temperature_list = [0.1]
strategies = [
    "Fact Statement",
    "Completion",
    # "Definitive Statement"
]

# Initialize a list to store the results
results = []
total_queries = 0
repeats = 5  # Number of times to repeat each generation

total_num_queries = len(model_names) * len(queries) * len(max_new_tokens_list) * len(temperature_list) * len(strategies) * repeats

for model_name in model_names:
    generator = ResponseGenerator(model_name)
    for query_idx, (query, true_answer) in enumerate(zip(queries, true_answers)):
        for max_new_tokens in max_new_tokens_list:
            for temperature in temperature_list:
                for strategy in strategies:
                    for run in range(repeats):
                        print(f"TOTAL: {total_queries + 1}/{total_num_queries}, MODEL: {model_name}, QUERY: {query_idx}, STRATEGY: {strategy}, MAX_NEW_TOKENS: {max_new_tokens}, TEMP: {temperature}, RUN: {run + 1}/{repeats}")
                        output_text, cum_prob, adj_prob, entropy, token_probs, is_correct = generator.generate_response(query, strategy, true_answer, max_new_tokens, temperature)
                        results.append({
                            "model": model_name,
                            "query": query,
                            "max_new_tokens": max_new_tokens,
                            "temp": temperature,
                            "strategy": strategy,
                            "true_answer": true_answer,
                            "output_text": output_text,
                            "is_correct": is_correct,
                            "cum_prob": round(cum_prob, 4),
                            "adj_prob": round(adj_prob.item(), 4),
                            "entropy": round(entropy, 4),
                            "token_probs": token_probs,  # Now includes tuples of (token, prob)
                            "run": run + 1  # Indicate which run this is
                        })
                        total_queries += 1
                        print(f"OUTPUT_TEXT: {output_text}, IS_CORRECT: {is_correct}, CUM_PROB: {cum_prob}")

# Convert the results list to a pandas DataFrame
df = pd.DataFrame(results)

# Print the DataFrame
print(df)

# Export the DataFrame to an Excel file
df.to_excel("multinomial_sampling_llama_5_runs-v2.xlsx", index=False)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.20it/s]


TOTAL: 1/240, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 15, TEMP: 0.1, RUN: 1/5
OUTPUT_TEXT: Simcoe Composite School is located at 1 Simcoe Street, Simcoe, IS_CORRECT: False, CUM_PROB: 1.0
TOTAL: 2/240, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 15, TEMP: 0.1, RUN: 2/5
OUTPUT_TEXT: Simcoe Composite School is located at 1 Simcoe Street, Simcoe, IS_CORRECT: False, CUM_PROB: 1.0
TOTAL: 3/240, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 15, TEMP: 0.1, RUN: 3/5
OUTPUT_TEXT: Simcoe Composite School is located at 1 Simcoe Street, Simcoe, IS_CORRECT: False, CUM_PROB: 1.0
TOTAL: 4/240, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 15, TEMP: 0.1, RUN: 4/5
OUTPUT_TEXT: Simcoe Composite School is located at 1 Simcoe Street, Simcoe, IS_CORRECT: False, CUM_PROB: 1.0
TOTAL: 5/240, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRAT

In [6]:
# Convert the results list to a pandas DataFrame
df = pd.DataFrame(results)

# Print the DataFrame
print(df)

# Export the DataFrame to an Excel file
df.to_excel("multinomial_sampling_llama_5_runs-v2.xlsx", index=False)

                         model  \
0   meta-llama/Meta-Llama-3-8B   
1   meta-llama/Meta-Llama-3-8B   
2   meta-llama/Meta-Llama-3-8B   
3   meta-llama/Meta-Llama-3-8B   
4   meta-llama/Meta-Llama-3-8B   
5   meta-llama/Meta-Llama-3-8B   
6   meta-llama/Meta-Llama-3-8B   
7   meta-llama/Meta-Llama-3-8B   
8   meta-llama/Meta-Llama-3-8B   
9   meta-llama/Meta-Llama-3-8B   
10  meta-llama/Meta-Llama-3-8B   
11  meta-llama/Meta-Llama-3-8B   
12  meta-llama/Meta-Llama-3-8B   
13  meta-llama/Meta-Llama-3-8B   
14  meta-llama/Meta-Llama-3-8B   
15  meta-llama/Meta-Llama-3-8B   
16  meta-llama/Meta-Llama-3-8B   
17  meta-llama/Meta-Llama-3-8B   
18  meta-llama/Meta-Llama-3-8B   
19  meta-llama/Meta-Llama-3-8B   
20  meta-llama/Meta-Llama-3-8B   
21  meta-llama/Meta-Llama-3-8B   
22  meta-llama/Meta-Llama-3-8B   
23  meta-llama/Meta-Llama-3-8B   
24  meta-llama/Meta-Llama-3-8B   
25  meta-llama/Meta-Llama-3-8B   
26  meta-llama/Meta-Llama-3-8B   
27  meta-llama/Meta-Llama-3-8B   
28  meta-llama

In [ ]:
df.groupby(["model", "strategy", "max_new_tokens", "temp"]).agg({
    "is_correct": ["sum"],
    "cum_prob": ["mean"]
}).to_excel("results_07_31_v2_agg.xlsx")

### 3.4 Beam + Multinomial

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

def get_prompt(query, strategy):
    if strategy == "Fact Statement":
        return f"{query} Fact:"
    elif strategy == "Completion":
        return f"{query} The answer is:"
    elif strategy == "Definitive Statement":
        return f"The answer to the question '{query}' is:"
    elif strategy == "Fill-in-the-Blank":
        return f"{query} is located in _____.\nAnswer:"
    elif strategy == "Structured Answer Prompt":
        return f"Question: {query}\nAnswer (one word):"
    elif strategy == "Direct Instruction":
        return f"Please answer the following question in one word.\nQuestion: {query}\nAnswer:"
    elif strategy == "Contextual Prompts":
        return f"{query} (Please answer in one word)"
    elif strategy == "Question-Answer Pairs":
        return f"QSTN: What is the capital of France?\nANSR: Paris\nQSTN: What is the capital of Germany?\nANSR: Berlin\nQSTN: {query}\nANSR:"
    elif strategy == "Direct Answer":
        return f"Please provide a short, direct answer to the following question: {query} Answer:"
    elif strategy == "Q&A Format":
        return f"Q: {query}\nA:"
    elif strategy == "Instructional":
        return f"Answer the following question in one or two words: {query}"
    elif strategy == "Summary":
        return f"Summarize the answer to the following question: {query}"
    elif strategy == "Echo":
        return f"{query} {query}"
    elif strategy == "True Completion":
        return f"{query} The true answer is:"
    elif strategy == "Direct Completion":
        return f"{query} Answer:"
    elif strategy == "Answer Completion":
        return f"{query} The correct answer is:"
    else:
        return query

def clean_response(output_text, strategy):
    if strategy == "Fact Statement":
        return output_text.split("Fact:")[-1].strip()
    elif strategy == "Completion":
        return output_text.split("The answer is:")[-1].strip()
    elif strategy == "Definitive Statement":
        return output_text.split("is:")[-1].strip()
    elif strategy == "Fill-in-the-Blank":
        return output_text.split("Answer:")[-1].strip()
    elif strategy == "Structured Answer Prompt":
        return output_text.split("Answer (one word):")[-1].strip()
    elif strategy == "Direct Instruction":
        return output_text.split("Answer:")[-1].strip()
    elif strategy == "Contextual Prompts":
        return output_text.split("Please answer in one word")[-1].strip()
    elif strategy == "Question-Answer Pairs":
        return output_text.split("ANSR:")[-1].strip()
    elif strategy == "Direct Answer":
        return output_text.split("Answer:")[-1].strip()
    elif strategy == "Q&A Format":
        return output_text.split("A:")[-1].strip()
    elif strategy == "Instructional":
        return output_text.split("Answer the following question")[-1].strip()
    elif strategy == "Summary":
        return output_text.split("Summarize the answer to the following question")[-1].strip()
    elif strategy == "Echo":
        return output_text
    elif strategy == "True Completion":
        return output_text.split("The true answer is:")[-1].strip()
    elif strategy == "Direct Completion":
        return output_text.split("Answer:")[-1].strip()
    elif strategy == "Answer Completion":
        return output_text.split("The correct answer is:")[-1].strip()
    else:
        return output_text.strip()

def calculate_entropy(probs):
    return -np.sum(probs * np.log(probs))

class ResponseGenerator:
    def __init__(self, model_name):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
        
        # Set pad_token_id to eos_token_id to avoid the warning
        self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
        self.model.eval()

    def generate_response(self, query, strategy, true_answer, max_new_tokens, temperature, use_beam_search, n_repeats=5, n_beams=5):
        prompt = get_prompt(query, strategy)
        inputs = self.tokenizer(prompt, return_tensors='pt').to("cuda")
        
        # Generation configuration
        generation_config = {
            "temperature": temperature,
            "do_sample": True,
            "top_p": 0.75,
            "top_k": 40,
            "output_scores": True,
            "output_hidden_states": False,
            "output_attentions": False,
            "return_dict_in_generate": True,
            "pad_token_id": self.tokenizer.eos_token_id
        }
        
        if use_beam_search:
            generation_config = {
                **generation_config,
                "num_beams": n_beams,
                "num_return_sequences": n_beams,
            }
            generation_config.pop("do_sample")
        
        results = []

        if use_beam_search:
            with torch.no_grad():
                outputs = self.model.generate(
                    inputs['input_ids'],
                    attention_mask=inputs['attention_mask'],  # Provide attention_mask to avoid the warning
                    generation_config=GenerationConfig(**generation_config),
                    max_new_tokens=max_new_tokens
                )
            transition_scores = self.model.compute_transition_scores(
                outputs.sequences,
                outputs.scores,
                normalize_logits=True,
                beam_indices=outputs.beam_indices
            )
            trans_scores = np.exp(transition_scores.cpu().numpy())

            for i in range(n_beams):  # Process each beam
                output_text = self.tokenizer.decode(outputs.sequences[i], skip_special_tokens=True)
                output_text = clean_response(output_text, strategy)

                token_probs = [(self.tokenizer.decode([outputs.sequences[i][j]]), round(trans_scores[i, j], 4))
                               for j in range(len(trans_scores[i]))]

                beam_prob = torch.exp(torch.sum(torch.log(torch.from_numpy(trans_scores[i])))).cpu().item()
                beam_prob_adj = torch.exp((len(trans_scores[0]) ** -1) * torch.sum(torch.log(torch.from_numpy(trans_scores[i])))).cpu().item()

                entropy = calculate_entropy(np.array(trans_scores[i]))
                is_correct = true_answer.lower() in output_text.lower()

                results.append({
                    "output_text": output_text,
                    "beam_prob": beam_prob,
                    "beam_prob_adj": beam_prob_adj,
                    "entropy": entropy,
                    "is_correct": is_correct,
                    "token_probs": token_probs,
                    "run": i + 1  # 1-indexed
                })

        else:
            for run in range(n_repeats):
                with torch.no_grad():
                    outputs = self.model.generate(
                        inputs['input_ids'],
                        attention_mask=inputs['attention_mask'],
                        generation_config=GenerationConfig(**generation_config),
                        max_new_tokens=max_new_tokens
                    )

                output_text = self.tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
                output_text = clean_response(output_text, strategy)

                response_tokens = outputs.sequences[0].tolist()
                probs = []
                sequence = outputs.sequences[0]
                shift_idx = len(sequence) - len(outputs.scores)  # Position to start processing tokens
                token_probs = []

                for pos_idx, beam_scores in enumerate(outputs.scores):
                    softmax_scores = torch.softmax(beam_scores, dim=-1)
                    token_id = sequence[pos_idx + shift_idx]
                    token_prob = softmax_scores[0][token_id].item()
                    token_probs.append((self.tokenizer.decode([token_id]), round(token_prob, 4)))
                    probs.append(token_prob)

                probs = torch.tensor(probs)
                beam_prob = torch.exp(torch.sum(torch.log(probs))).item()
                beam_prob_adj = torch.exp((len(probs) ** -1) * torch.sum(torch.log(probs))).item()

                entropy = calculate_entropy(np.array(probs))
                is_correct = true_answer.lower() in output_text.lower()

                results.append({
                    "output_text": output_text,
                    "beam_prob": beam_prob,
                    "beam_prob_adj": beam_prob_adj,
                    "entropy": entropy,
                    "is_correct": is_correct,
                    "token_probs": token_probs,
                    "run": run + 1  # 1-indexed
                })

        return results

# Example usage:
model_names = [
    #"bigscience/bloomz-560m",
    "meta-llama/Meta-Llama-3-8B",
    #"meta-llama/Meta-Llama-3-8B-Instruct",
    "bigscience/bloomz-1b1",
    "openai-community/gpt2-large",
    #"EleutherAI/gpt-neo-1.3B",
    "TinyLlama/TinyLlama_v1.1",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
]
queries_and_answers = [
    # Place / Region Names (3)
    ("What is the location of the Great Wall of China?", "China"),
    ("Where is the Eiffel Tower located?", "Paris"),
    ("What is the capital city of Japan?", "Tokyo"),
    
    # Adjectives (3)
    ("How would you describe the taste of a lemon?", "Sour"),
    ("What is the best way to describe the weather on a clear day?", "Sunny"),
    ("How would you describe the feeling of touching velvet?", "Soft"),
    
    # Person Names (3)
    ("Who was the first president of the United States?", "George Washington"),
    ("Who is the author of 'Pride and Prejudice'?", "Jane Austen"),
    ("Who painted the Mona Lisa?", "Leonardo da Vinci"),
    
    # Abstract Answers (3)
    ("What is your favorite color?", "I don't know"),
    ("What do you think of the meaning of life?", "It's complicated"),
    ("What comes after the end?", "Nothing"),
    
    # Numerical Answers (3)
    ("How many continents are there on Earth?", "Seven"),
    ("What is the boiling point of water in Celsius?", "100"),
    ("How many hours are there in a day?", "24"),
    
    # Languages (3)
    ("What language is primarily spoken in Brazil?", "Portuguese"),
    ("Which language is spoken in Germany?", "German"),
    ("What is the official language of China?", "Mandarin"),
    
    # Company Names (3)
    ("Which company developed the iPhone?", "Apple"),
    ("What is the name of the online retailer founded by Jeff Bezos?", "Amazon"),
    ("Which company is known for its search engine?", "Google")
]

# max_new_tokens_list = [10, 15, 20, 30, 40]
max_new_tokens_list = [10, 15, 20, 30, 40]
temperature_list = [0.1]
strategies = [
    "Fact Statement",
    "Completion",
    "Definitive Statement",
    "Fill-in-the-Blank",
    "Structured Answer Prompt",
    "Direct Instruction",
    "Contextual Prompts",
    "Question-Answer Pairs",
    "Direct Answer",
    "Q&A Format",
    "Instructional",
    "Summary",
    "Echo",
    "True Completion",
    "Direct Completion",
    "Answer Completion"
]

n_repeats = 5
use_beam_search_list = [True, False]

# Initialize a list to store the results
results = []

n_steps = 0
total_steps = len(model_names) * len(queries_and_answers) * len(max_new_tokens_list) * len(temperature_list) * len(strategies) * n_repeats * len(use_beam_search_list)
for model_name in model_names:
    generator = ResponseGenerator(model_name)
    for query_idx, (query, true_answer) in enumerate(queries_and_answers):
        for max_new_tokens in max_new_tokens_list:
            for temperature in temperature_list:
                for strategy in strategies:
                    for use_beam_search in use_beam_search_list:  # Iterate over Beam Search and Multinomial Sampling
                        run_results = generator.generate_response(query, strategy, true_answer, max_new_tokens, temperature, use_beam_search, n_repeats=n_repeats, n_beams=n_repeats)
                        for result_dict in run_results:
                            print(f"TOTAL: {n_steps + 1}/{total_steps}, MODEL: {model_name}, QUERY: {query_idx}, STRATEGY: {strategy}, MAX_NEW_TOKENS: {max_new_tokens}, RUN: {result_dict['run']}/{n_repeats}")
                            # Store the results in the list
                            results.append({
                                "Model": model_name,
                                "Query": query,
                                "Strategy": strategy,
                                "Max New Tokens": max_new_tokens,
                                "Temperature": temperature,
                                "Beam Search": use_beam_search,
                                "Run": result_dict['run'],
                                "Generated Response": result_dict['output_text'],
                                "P": result_dict['beam_prob'],
                                "P_adj": result_dict['beam_prob_adj'],
                                "Entropy": result_dict['entropy'],
                                "Is Correct": result_dict['is_correct'],
                                "Token Probabilities": result_dict['token_probs']
                            })
                            print(f"IS_CORRECT: {result_dict['is_correct']}, PROB: {result_dict['beam_prob']}, ADJ_PROB: {result_dict['beam_prob_adj']}, ENTROPY: {result_dict['entropy']}")
                            n_steps += 1
                            if n_steps >= 100:
                                # pass

                                df_results = pd.DataFrame(results)
                                df_results.to_excel("beam_multinomial_llama_08_21_test-raw_table.xlsx", index=False)

                                df_results = pd.DataFrame(results)

                                def calculate_scores(group):
                                    y_true = group['Is Correct'].values
                                    y_scores = group['P'].values
                                    
                                    # Ensure at least two classes are present for AUROC
                                    if len(set(y_true)) > 1:
                                        aucroc = roc_auc_score(y_true, y_scores)
                                    else:
                                        aucroc = np.nan
                                    
                                    aucpr = average_precision_score(y_true, y_scores)
                                    accuracy = np.mean(y_true)
                                    
                                    return pd.Series({'AUROC Score': aucroc, 'AUCPR Score': aucpr, 'Accuracy': accuracy})

                                # Group by the relevant columns and calculate AUROC, AUCPR, and accuracy for each group
                                df_scores = df_results.groupby(['Model', 'Max New Tokens', 'Temperature', 'Strategy', 'Beam Search']).apply(calculate_scores).reset_index()
                                df_results = df_results.merge(df_scores, on=['Model', 'Max New Tokens', 'Temperature', 'Strategy', 'Beam Search'])

                                # Save the original detailed results to an Excel file
                                df_results.to_excel("beam_multinomial_llama_08_21_test-table.xlsx", index=False)
                                df_scores.to_excel("beam_multinomial_llama_08_21_test-scores.xlsx", index=False)
                                
                                raise KeyboardInterrupt


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.02s/it]
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:515: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:520: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.75` -- this flag is only used in sample-based generation modes. You should set `do_sample=T

TOTAL: 1/84000, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/5
IS_CORRECT: True, PROB: 0.007930183783173561, ADJ_PROB: 0.6164932250976562, ENTROPY: 2.0676867961883545
TOTAL: 2/84000, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 2/5
IS_CORRECT: True, PROB: 0.005730344448238611, ADJ_PROB: 0.5967852473258972, ENTROPY: 1.9738126993179321
TOTAL: 3/84000, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 3/5
IS_CORRECT: True, PROB: 0.004067429341375828, ADJ_PROB: 0.5766761898994446, ENTROPY: 2.3833987712860107
TOTAL: 4/84000, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 4/5
IS_CORRECT: True, PROB: 0.003088973928242922, ADJ_PROB: 0.5610239505767822, ENTROPY: 2.336489200592041
TOTAL: 5/84000, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 5/5
IS_CO

/tmp/ipykernel_181463/3452781674.py:187: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  entropy = calculate_entropy(np.array(probs))


TOTAL: 6/84000, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 1/5
IS_CORRECT: True, PROB: 1.0, ADJ_PROB: 1.0, ENTROPY: -0.0
TOTAL: 7/84000, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 2/5
IS_CORRECT: True, PROB: 1.0, ADJ_PROB: 1.0, ENTROPY: -0.0
TOTAL: 8/84000, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 3/5
IS_CORRECT: True, PROB: 1.0, ADJ_PROB: 1.0, ENTROPY: -0.0
TOTAL: 9/84000, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 4/5
IS_CORRECT: True, PROB: 1.0, ADJ_PROB: 1.0, ENTROPY: -0.0
TOTAL: 10/84000, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Fact Statement, MAX_NEW_TOKENS: 10, RUN: 5/5
IS_CORRECT: True, PROB: 1.0, ADJ_PROB: 1.0, ENTROPY: -0.0
TOTAL: 11/84000, MODEL: meta-llama/Meta-Llama-3-8B, QUERY: 0, STRATEGY: Completion, MAX_NEW_TOKENS: 10, RUN: 1/5
IS_CORRECT: True

/tmp/ipykernel_181463/3452781674.py:332: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_scores = df_results.groupby(['Model', 'Max New Tokens', 'Temperature', 'Strategy', 'Beam Search']).apply(calculate_scores).reset_index()


KeyboardInterrupt: 

In [ ]:
df_results = pd.DataFrame(results)

def calculate_scores(group):
    y_true = group['Is Correct'].values
    y_scores = group['P'].values
    
    # Ensure at least two classes are present for AUROC
    if len(set(y_true)) > 1:
        aucroc = roc_auc_score(y_true, y_scores)
    else:
        aucroc = np.nan
    
    aucpr = average_precision_score(y_true, y_scores)
    accuracy = np.mean(y_true)
    
    return pd.Series({'AUROC Score': aucroc, 'AUCPR Score': aucpr, 'Accuracy': accuracy})

# Group by the relevant columns and calculate AUROC, AUCPR, and accuracy for each group
df_scores = df_results.groupby(['Model', 'Max New Tokens', 'Temperature', 'Strategy', 'Beam Search']).apply(calculate_scores).reset_index()
df_results = df_results.merge(df_scores, on=['Model', 'Max New Tokens', 'Temperature', 'Strategy', 'Beam Search'])

# Save the original detailed results to an Excel file
df_results.to_excel("beam_multinomial_llama_table_08_21.xlsx", index=False)
df_scores.to_excel("beam_multinomial_llama_scores_08_21.xlsx", index=False)

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1030: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1030: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1030: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:1030: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:10

In [7]:
df_scores

,Model,Max New Tokens,Temperature,Strategy,Beam Search,AUROC Score,AUCPR Score,Accuracy
0,meta-llama/Meta-Llama-3-8B,15,0.1,Completion,False,NaN,-0.000000,0.0
1,meta-llama/Meta-Llama-3-8B,15,0.1,Completion,True,NaN,-0.000000,0.0
2,meta-llama/Meta-Llama-3-8B,15,0.1,Contextual Prompts,False,NaN,-0.000000,0.0
3,meta-llama/Meta-Llama-3-8B,15,0.1,Contextual Prompts,True,0.250000,0.250000,0.2
4,meta-llama/Meta-Llama-3-8B,15,0.1,Definitive Statement,False,NaN,1.000000,1.0
5,meta-llama/Meta-Llama-3-8B,15,0.1,Definitive Statement,True,0.250000,0.804167,0.8
6,meta-llama/Meta-Llama-3-8B,15,0.1,Direct Answer,False,NaN,-0.000000,0.0
7,meta-llama/Meta-Llama-3-8B,15,0.1,Direct Answer,True,0.333333,0.500000,0.4
8,meta-llama/Meta-Llama-3-8B,15,0.1,Direct Instruction,False,NaN,1.000000,1.0
9,meta-llama/Meta-Llama-3-8B,15,0.1,Direct Instruction,True,0.166667,0.366667,0.4


In [6]:
df_results

,Model,Query,Strategy,Max New Tokens,Temperature,Beam Search,Run,Generated Response,P,P_adj,Entropy,Is Correct,Token Probabilities,AUROC Score,AUCPR Score,Accuracy
0,meta-llama/Meta-Llama-3-8B,What is the location of Simcoe Composite School?,Fact Statement,15,0.1,True,1,"Simcoe Composite School is located in Simcoe, ...",0.000542,0.605737,3.293906,True,"[(<|begin_of_text|>, 0.3303), (What, 0.9974), ...",0.25,0.804167,0.8
1,meta-llama/Meta-Llama-3-8B,What is the location of Simcoe Composite School?,Fact Statement,15,0.1,True,2,"Simcoe Composite School is located in Simcoe, ...",0.000474,0.600333,3.114716,False,"[(<|begin_of_text|>, 0.3303), (What, 0.9974), ...",0.25,0.804167,0.8
2,meta-llama/Meta-Llama-3-8B,What is the location of Simcoe Composite School?,Fact Statement,15,0.1,True,3,"Simcoe Composite School is located in Simcoe, ...",0.000378,0.591309,3.226967,True,"[(<|begin_of_text|>, 0.3303), (What, 0.9974), ...",0.25,0.804167,0.8
3,meta-llama/Meta-Llama-3-8B,What is the location of Simcoe Composite School?,Fact Statement,15,0.1,True,4,"Simcoe Composite School is located in Simcoe, ...",0.000276,0.579117,3.184928,True,"[(<|begin_of_text|>, 0.3303), (What, 0.9974), ...",0.25,0.804167,0.8
4,meta-llama/Meta-Llama-3-8B,What is the location of Simcoe Composite School?,Fact Statement,15,0.1,True,5,"Simcoe Composite School is located in Simcoe, ...",0.000256,0.576195,3.223548,True,"[(<|begin_of_text|>, 0.3303), (What, 0.9974), ...",0.25,0.804167,0.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,meta-llama/Meta-Llama-3-8B,What is the location of Simcoe Composite School?,Q&A Format,15,0.1,False,1,Simcoe Composite School is located at 1000 Sim...,0.500000,0.954842,0.346574,False,"[( Sim, 1.0), (coe, 1.0), ( Composite, 1.0), (...",NaN,-0.000000,0.0
96,meta-llama/Meta-Llama-3-8B,What is the location of Simcoe Composite School?,Q&A Format,15,0.1,False,2,Simcoe Composite School is located at 1000 Sim...,0.500000,0.954842,0.346574,False,"[( Sim, 1.0), (coe, 1.0), ( Composite, 1.0), (...",NaN,-0.000000,0.0
97,meta-llama/Meta-Llama-3-8B,What is the location of Simcoe Composite School?,Q&A Format,15,0.1,False,3,Simcoe Composite School is located at 1000 Sim...,0.500000,0.954842,0.346574,False,"[( Sim, 1.0), (coe, 1.0), ( Composite, 1.0), (...",NaN,-0.000000,0.0
98,meta-llama/Meta-Llama-3-8B,What is the location of Simcoe Composite School?,Q&A Format,15,0.1,False,4,Simcoe Composite School is located at 1000 Sim...,0.500000,0.954842,0.346574,False,"[( Sim, 1.0), (coe, 1.0), ( Composite, 1.0), (...",NaN,-0.000000,0.0


In [ ]:
df.to_excel("results_07_25.xlsx", index=False)

In [ ]:
df.iloc[12, :]

In [ ]:
df.loc[df['is_correct'] == False]['output_text']